# Notebook 01 — 3D Source Localization with Spatial LibriSpeech

---

## Overview

This notebook is **Part 1** of a multi-part evaluation pipeline that benchmarks a custom **Bicomplex-PyTorch** neural network against standard real-valued architectures on the task of **3D Speech Source Localization**.

In this first notebook we:
1. Describe the **Spatial LibriSpeech** dataset and its parquet schema.
2. Explain the 3D Source Localization objective and why Ambisonics recordings make bicomplex arithmetic a natural fit.
3. Instantiate the `SpatialLibriSpeechDataset` PyTorch `Dataset` object and inspect a sample.

---

## 1. The Spatial LibriSpeech Dataset

**Spatial LibriSpeech** (Apple ML Research, 2022) is a large-scale, simulated spatial audio dataset built on top of the well-known LibriSpeech corpus. It provides **4-channel first-order Ambisonics (FOA)** recordings of speech and distractor-noise sources placed at arbitrary 3D positions inside simulated rooms.

### Key numbers

| Property | Value |
|---|---|
| Total samples | ~22 k |
| Audio format | 4-channel FOA `.flac`, 16 kHz |
| Splits | `train` / `test` |
| Metadata format | Apache Parquet |
| Metadata URL | `https://docs-assets.developer.apple.com/ml-research/datasets/spatial-librispeech/v1/metadata.parquet` |

---

### 1.1 Parquet Metadata Schema

The metadata file is a flat Parquet table where each row describes one sample. The schema is reproduced below, grouped by category.

#### Sample identification

| Column | Type | Range / Values | Description |
|---|---|---|---|
| `sample_id` | `int` | [0, 22 k] | Unique sample identifier. Audio filename: `{sample_id:06d}.flac` |
| `split` | `str` | `train`, `test` | Dataset split |
| `lite_version` | `bool` | — | Whether this sample belongs to the lite subset |

#### Speech targets — the 3D localization labels

| Column | Type | Unit | Description |
|---|---|---|---|
| `speech/azimuth` | `float` | radians | **Horizontal** angle from microphone array to speech source |
| `speech/elevation` | `float` | radians | **Vertical** angle from microphone array to speech source |
| `speech/distance` | `float` | metres | Distance between source and array |
| `speech/speaking_azimuth` | `float` | radians | Horizontal rotation of the source (0 = facing the array) |
| `speech/speaking_elevation` | `float` | radians | Vertical rotation of the source (0 = facing the array) |
| `speech/source_id` | `float` | [1, 20] | Discrete source position identifier |
| `speech/directivity_id` | `int` | [0, 15] | Head-related directivity profile |
| `speech/mrp` | `float` | dB-ASL | Signal power at the mouth reference point |

#### Audio information

| Column | Type | Unit | Description |
|---|---|---|---|
| `audio_info/duration` | `float` | seconds | Sample duration |
| `audio_info/frames` | `int` | — | Number of frames at 16 kHz |
| `audio_info/size/ambisonics` | `int` | bytes | File size of the FOA `.flac` |
| `audio_info/size/noise_ambisonics` | `int` | bytes | File size of the noise `.flac` |
| `audio_info/checksum/ambisonics` | `str` | hex | SHA-256 checksum of the FOA recording |
| `audio_info/checksum/noise_ambisonics` | `str` | hex | SHA-256 checksum of the noise recording |

#### Acoustics (33-element arrays, one value per third-octave bin)

| Column | Unit | Description |
|---|---|---|
| `acoustics/frequency_bins` | Hz | Mean frequency of each third-octave bin |
| `acoustics/c50_db` | dB | Speech clarity C50 |
| `acoustics/drr_db` | dB | Direct-to-reverberant ratio (DRR) |
| `acoustics/edt_ms` | ms | Early decay time (EDT) |
| `acoustics/t20_ms` | ms | 20 dB decay duration (T20) |
| `acoustics/t30_ms` | ms | 30 dB decay duration (T30) |

#### Distractor noise

| Column | Type | Unit | Description |
|---|---|---|---|
| `noise/azimuth` | `float` | radians | Horizontal angle to distractor noise source |
| `noise/elevation` | `float` | radians | Vertical angle to distractor noise source |
| `noise/distance` | `float` | metres | Distance to noise source |
| `noise/snr` | `float` | dB | Signal-to-noise ratio |
| `noise/source_id` | `float` | [1, 20] | Noise source position identifier |

#### Room

| Column | Type | Unit | Description |
|---|---|---|---|
| `room/room_id` | `int` | — | Simulated room identifier |
| `room/volume` | `float` | m³ | Room volume |
| `room/floor_area` | `float` | m² | Floor area |
| `room/surface_area` | `float` | m² | Total surface area |

#### Original LibriSpeech metadata

| Column | Type | Description |
|---|---|---|
| `speech/librispeech_metadata/reader_id` | `int` | Numeric reader identifier |
| `speech/librispeech_metadata/reader_name` | `str` | Reader alias |
| `speech/librispeech_metadata/reader_sex` | `str` | `m` or `f` |
| `speech/librispeech_metadata/transcription` | `str` | Text being read |
| `speech/librispeech_metadata/subset` | `str` | Original LibriSpeech subset |

---

## 2. The 3D Source Localization Task

### 2.1 Objective

Given a 4-channel FOA waveform of duration $T$ samples at 16 kHz, predict the **azimuth** $\phi \in (-\pi, \pi]$ and **elevation** $\theta \in [-\pi/2, \pi/2]$ of the primary speech source — both in radians.

This is a **continuous regression** problem. Common evaluation metrics:

| Metric | Formula | Notes |
|---|---|---|
| MAE | $\frac{1}{N}\sum|\hat{y}-y|$ | Per angle |
| Angular Error | geodesic distance on unit sphere | Handles wraparound |
| RMSE | $\sqrt{\frac{1}{N}\sum(\hat{y}-y)^2}$ | Penalises outliers |

### 2.2 Why First-Order Ambisonics?

FOA encodes the full 3D sound field into four channels using spherical harmonics:

| Channel | Harmonic | Physical interpretation |
|---|---|---|
| W | $Y_0^0$ | Omnidirectional pressure (monopole) |
| X | $Y_1^{-1}$ | Figure-of-eight, front–back axis |
| Y | $Y_1^1$ | Figure-of-eight, left–right axis |
| Z | $Y_1^0$ | Figure-of-eight, up–down axis |

The directional information is encoded entirely in the **phase relationships** between these four channels. Standard real-valued convolutions process each channel independently and may discard this cross-channel coherence. Bicomplex arithmetic preserves it by design.

### 2.3 Why Bicomplex Networks?

A bicomplex number $q = a + b\,\mathbf{i}_1 + c\,\mathbf{i}_2 + d\,\mathbf{i}_1\mathbf{i}_2$ is a 4-tuple of reals with a rich algebraic structure. Mapping $(W, X, Y, Z) \to q$ treats the FOA quartet as a **single algebraic entity**:

* **Inter-channel phase coherence** is maintained through bicomplex multiplication.
* **Parameter efficiency**: a bicomplex convolution weight matrix operates on the full 4-tuple rather than four independent real scalars.
* **Geometric equivariance**: rotation of the sound field corresponds to structured operations in bicomplex space.

---

## 3. The `SpatialLibriSpeechDataset` Class

### 3.1 Design Principles

| Principle | Implementation |
|---|---|
| Single responsibility | Loading, filtering, path-building, and I/O are separate private methods |
| Lazy audio loading | Waveforms loaded on demand in `__getitem__`; only the DataFrame lives in RAM |
| Transform pipeline | `transform` and `target_transform` callables mirror the `torchvision` API |
| Data integrity | Optional SHA-256 checksum verification |
| Transparency | `__repr__` and `summary()` make the active configuration inspectable |

### 3.2 Constructor Signature

```python
SpatialLibriSpeechDataset(
    audio_root: str | Path,           # directory containing the .flac files
    metadata_path: str | Path = URL,  # local .parquet path or remote URL
    split: 'train' | 'test' = 'train',
    lite_version_only: bool = False,  # restrict to lite subset
    transform: Callable | None = None,         # applied to waveform (4, T)
    target_transform: Callable | None = None,  # applied to (azimuth, elevation)
    verify_checksums: bool = False,
    sample_rate: int = 16_000,
)
```

### 3.3 `__getitem__` Return Type

```
(waveform, azimuth, elevation)
  waveform  : torch.Tensor  shape (4, T)  float32  — 4-ch FOA waveform
  azimuth   : torch.Tensor  scalar        float32  — radians
  elevation : torch.Tensor  scalar        float32  — radians
```

In [ ]:
# ── Stdlib ───────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

# ── Add examples/audio_processing to sys.path so 'src' is importable ────────
AUDIO_PROCESSING_DIR = Path("../").resolve()
if str(AUDIO_PROCESSING_DIR) not in sys.path:
    sys.path.insert(0, str(AUDIO_PROCESSING_DIR))

print(f"Inserted into sys.path: {AUDIO_PROCESSING_DIR}")

In [ ]:
# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

# ── Local ─────────────────────────────────────────────────────────────────────
from src.dataset import SpatialLibriSpeechDataset, METADATA_URL

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
print(f"Metadata : {METADATA_URL}")

### 3.4 Configure Paths

Edit the two variables below before running:

* **`AUDIO_ROOT`** — local directory containing the `.flac` files.
* **`METADATA_PATH`** — either a local `.parquet` path or the default URL to stream from Apple's CDN.

Expected `AUDIO_ROOT` layout:
```
AUDIO_ROOT/
├── 000000.flac   # 4-channel FOA, 16 kHz
├── 000001.flac
└── ...
```

In [ ]:
# ── User configuration ────────────────────────────────────────────────────────
AUDIO_ROOT        = Path("/data/spatial_librispeech/ambisonics")
METADATA_PATH     = METADATA_URL   # or Path("/data/spatial_librispeech/metadata.parquet")

SPLIT             = "train"    # "train" | "test"
LITE_VERSION_ONLY = False      # True → use only the lite subset
BATCH_SIZE        = 8
NUM_WORKERS       = 4

### 3.5 Instantiate the Dataset

The constructor performs these steps **synchronously**:

1. Validates `split` ∈ `{"train", "test"}` and confirms `audio_root` exists.
2. Loads (or streams) the Parquet metadata into a `pandas.DataFrame`.
3. Filters rows to the requested `split` (and optionally `lite_version == True`).
4. Resets the integer index so that positional lookup in `__getitem__` is O(1).

No audio files are opened at this stage — all I/O is deferred to `__getitem__`.

In [ ]:
dataset = SpatialLibriSpeechDataset(
    audio_root=AUDIO_ROOT,
    metadata_path=METADATA_PATH,
    split=SPLIT,
    lite_version_only=LITE_VERSION_ONLY,
    verify_checksums=False,   # set True for data integrity validation
    sample_rate=16_000,
)

print(dataset)
print(f"\nTotal samples in '{SPLIT}' split: {len(dataset):,}")

### 3.6 Inspect Metadata Summary

In [ ]:
summary = dataset.summary()
print(f"Summary shape: {summary.shape}")
summary.head(10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(
    np.degrees(summary["speech/azimuth"]),
    bins=36, color="steelblue", edgecolor="white",
)
axes[0].set_xlabel("Azimuth (degrees)")
axes[0].set_ylabel("Count")
axes[0].set_title(f"Azimuth distribution — {SPLIT} split")

axes[1].hist(
    np.degrees(summary["speech/elevation"]),
    bins=18, color="tomato", edgecolor="white",
)
axes[1].set_xlabel("Elevation (degrees)")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Elevation distribution — {SPLIT} split")

plt.tight_layout()
plt.show()

### 3.7 Retrieve a Single Sample

Calling `dataset[i]` triggers `__getitem__`, which:
1. Looks up row `i` in the filtered DataFrame (O(1) via `iloc`).
2. Extracts `speech/azimuth` and `speech/elevation` as scalar `float32` tensors.
3. Builds the file path `{audio_root}/{sample_id:06d}.flac`.
4. Loads the 4-channel FOA waveform with `torchaudio.load` → shape `(4, T)`, dtype `float32`.
5. Applies optional `transform` / `target_transform`.
6. Returns `(waveform, azimuth, elevation)`.

In [ ]:
waveform, azimuth, elevation = dataset[0]

print(f"Waveform shape : {waveform.shape}   (channels, time-steps)")
print(f"Waveform dtype : {waveform.dtype}")
print(f"Azimuth        : {azimuth.item():.4f} rad  ({np.degrees(azimuth.item()):.1f}°)")
print(f"Elevation      : {elevation.item():.4f} rad  ({np.degrees(elevation.item()):.1f}°)")
print(f"Audio path     : {dataset.get_audio_path(0)}")

In [ ]:
# Plot 1 second of all four FOA channels
channel_labels = ["W (omni)", "X (front–back)", "Y (left–right)", "Z (up–down)"]
one_second = 16_000
t = np.arange(one_second) / 16_000

fig, axes = plt.subplots(4, 1, figsize=(14, 8), sharex=True)
for ch, (ax, label) in enumerate(zip(axes, channel_labels)):
    ax.plot(t, waveform[ch, :one_second].numpy(), linewidth=0.5)
    ax.set_ylabel(label, fontsize=9)
    ax.set_ylim(-1, 1)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    f"FOA channels — sample #{dataset.get_sample_id(0):06d}  "
    f"az={np.degrees(azimuth.item()):.1f}°  "
    f"el={np.degrees(elevation.item()):.1f}°",
    fontsize=11,
)
plt.tight_layout()
plt.show()

### 3.8 Create a DataLoader

`SpatialLibriSpeechDataset` is a standard `torch.utils.data.Dataset` subclass and is fully compatible with `DataLoader` for batched, multi-worker loading.

In [ ]:
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=(SPLIT == "train"),
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print(f"DataLoader: {len(loader)} batches of size {BATCH_SIZE}")

batch_wav, batch_az, batch_el = next(iter(loader))
print(f"\nBatch waveforms : {batch_wav.shape}")
print(f"Batch azimuths  : {batch_az.shape}")
print(f"Batch elevations: {batch_el.shape}")

---

## Part 2 — Feature Engineering: Active & Reactive Acoustic Intensity

---

### 4.1 Physical Justification

#### Why not use raw waveforms or spectrograms?

A raw spectrogram of a single channel discards the *directional* information
that is encoded in the **phase relationships** between the four B-format
channels.  Simply concatenating four spectrograms along the channel axis
forces the network to rediscover those relationships from scratch.  Instead,
we pre-compute physically meaningful cross-channel features that make the
direction of arrival (DOA) directly accessible to the first layer of the
network.

#### Acoustic intensity (Jacobsen & Rydén, 1990)

The **acoustic intensity vector** $\\mathbf{I}(t)$ describes the instantaneous
net flow of acoustic energy at the microphone:

$$\\mathbf{I}(t) = p(t)\\,\\mathbf{u}(t)$$

where $p(t)$ is the sound pressure and $\\mathbf{u}(t)$ is the particle
velocity vector.  In a first-order Ambisonics (FOA) recording the four
B-format channels map directly onto these quantities:

| B-format channel | Physical quantity |
|---|---|
| W (omni) | $p$ — omnidirectional pressure (monopole) |
| X | $u_x$ — particle velocity, front–back axis |
| Y | $u_y$ — particle velocity, left–right axis |
| Z | $u_z$ — particle velocity, up–down axis |

Decomposing into time-frequency bins via the STFT and applying the
**cross-spectrum** estimator gives two physically distinct quantities:

$$\\underbrace{I_{a,d}(f,t) = \\operatorname{Re}\\bigl[W(f,t)\\cdot X_d(f,t)^*\\bigr]}_{\\text{Active intensity (net power flow)}}$$

$$\\underbrace{I_{r,d}(f,t) = \\operatorname{Im}\\bigl[W(f,t)\\cdot X_d(f,t)^*\\bigr]}_{\\text{Reactive intensity (stored / circulating energy)}}$$

for direction axis $d \\in \\{x, y, z\\}$, where $*$ denotes complex conjugation.

**Active intensity** points from the microphone array *toward* the dominant
source.  Its direction provides a direct estimate of the DOA:

$$\\hat{\\phi} = \\arctan2(I_{a,y},\\, I_{a,x}), \\qquad
\\hat{\\theta} = \\arcsin\\!\\left(\\frac{I_{a,z}}{\\|\\mathbf{I}_a\\|}\\right)$$

**Reactive intensity** captures the phase quadrature component —
useful in reverberant environments where reflected energy circles the array.
Together, the six components $(I_{a,x},\, I_{a,y},\, I_{a,z},\,
I_{r,x},\, I_{r,y},\, I_{r,z})$ form a compact, physics-informed
representation that encodes both DOA and room-acoustic characteristics.

#### W-power normalisation

Dividing each component by $|W(f,t)|^2 + \\varepsilon$ removes the
dependence on source level, yielding a unit DOA vector per time-frequency
bin that is robust to loudness variation and distant sources:

$$\\tilde{I}_{a,d}(f,t) = \\frac{I_{a,d}(f,t)}{|W(f,t)|^2 + \\varepsilon}$$

---

### 4.2 STFT Configuration

| Parameter | Value | Rationale |
|---|---|---|
| `n_fft` | 512 | 32 ms at 16 kHz — resolves key speech formants |
| `hop_length` | 128 | 8 ms (75 % overlap) — fine temporal resolution |
| `win_length` | 512 | Equal to `n_fft`; standard for speech tasks |
| `window` | Hann | Good sidelobe rejection; universal default |
| `center` | True | Frame 0 centred at sample 0; avoids boundary bias |
| `F` (freq bins) | 257 | `n_fft // 2 + 1`, one-sided real FFT |

At 16 kHz with these settings, a 0.5 s chunk yields
approximately **63 STFT frames**, giving a feature map of shape
``(6, 257, 63)`` per chunk.

---

### 4.3 Hardware-Optimised Chunking Strategy (16 GB VRAM Budget)

Spatial LibriSpeech utterances vary from roughly 1 s to 30 s.  Loading
a full variable-length recording onto the GPU would:

* Make collation in the `DataLoader` impossible without padding to the
  longest sequence in each batch (padding ratio can exceed 10×).
* Cause the GPU memory footprint to fluctuate wildly between batches,
  triggering CUDA out-of-memory errors on a 16 GB card.
* Bias gradient updates toward longer utterances.

**Solution — CPU chunking before collation:**

```
Disk
 └─ torchaudio.load()          (4, T)            CPU  ← variable T
     └─ chunk_waveform()       (N, 4, 8000)      CPU  ← fixed size
         └─ compute_stft()     (N, 4, 257, 63)   CPU  ← complex
             └─ extract_ar()   (N, 6, 257, 63)   CPU  ← float32
                 └─ DataLoader collate            GPU  ← batch of chunks
```

Each `__getitem__` call returns a tensor of shape ``(N_chunks, 6, 257, 63)``.
The `DataLoader` collates these into ``(B * N_chunks, 6, 257, 63)``
where `B` is the mini-batch size.  **All** feature computation happens on
the CPU workers; the GPU sees only fixed-size tensors.

#### Memory budget calculation

| Quantity | Value |
|---|---|
| Chunk feature shape | `(6, 257, 63)` |
| Bytes per chunk | `6 × 257 × 63 × 4 = 388,584 bytes ≈ 0.37 MB` |
| Batch size = 32 chunks | `≈ 11.9 MB` |
| Model activations (est.) | `~1–2 GB` |
| Gradients (est.) | `~1–2 GB` |
| **Total (safe budget)** | **< 6 GB**, well within 16 GB |

Adjusting `batch_size` or `chunk_duration` gives a simple knob to trade
off memory versus training speed.

In [ ]:
from src.preprocess import ActiveReactivePreprocessor

preprocessor = ActiveReactivePreprocessor(
    sample_rate=16_000,
    chunk_duration=0.5,      # seconds → 8 000 samples per chunk
    n_fft=512,               # 32 ms window → F = 257 frequency bins
    hop_length=128,          # 8 ms hop → ~63 frames per 0.5 s chunk
    win_length=512,
    window_fn="hann",
    center=True,
    normalize_intensity=True,
    drop_last_chunk=False,   # keep the zero-padded final chunk
)

print(preprocessor)
print(f"\nOutput channels : {preprocessor.output_channels}")
print(f"Feature names   : {preprocessor.feature_names}")
print(f"Freq bins (F)   : {preprocessor.n_freq}")

In [ ]:
# ── Apply to the first sample loaded in Part 1 ───────────────────────────────
# `waveform` was retrieved earlier as dataset[0][0]: shape (4, T)
assert waveform.device.type == "cpu", "Chunking must happen on CPU"

print(f"Input waveform shape : {tuple(waveform.shape)}  (channels, samples)")
print(f"Expected output      : {preprocessor.output_shape(waveform.shape[-1])}")
print()

# ── Stage 1: chunk ───────────────────────────────────────────────────────────
chunks = preprocessor.chunk_waveform(waveform)
print(f"After chunk_waveform : {tuple(chunks.shape)}  (N_chunks, channels, chunk_samples)")

# ── Stage 2: STFT ────────────────────────────────────────────────────────────
stft = preprocessor.compute_stft(chunks)
print(f"After compute_stft   : {tuple(stft.shape)}  (N_chunks, channels, F, T_frames)")
print(f"  dtype              : {stft.dtype}")

# ── Stage 3: active / reactive extraction ────────────────────────────────────
features = preprocessor.extract_active_reactive(stft)
print(f"After extract_ar     : {tuple(features.shape)}  (N_chunks, 6, F, T_frames)")
print(f"  dtype              : {features.dtype}")
print()

# ── Full pipeline (equivalent to the three stages above) ─────────────────────
features_full = preprocessor(waveform)
assert torch.allclose(features, features_full), "Pipeline mismatch!"
print(f"Full pipeline output : {tuple(features_full.shape)}")
print(f"Min / Max            : {features_full.min():.4f} / {features_full.max():.4f}")
print(f"Memory (float32)     : {features_full.numel() * 4 / 1024:.1f} KB")

In [ ]:
# ── Visualise the 6 intensity feature maps for chunk 0 ───────────────────────
chunk_idx = 0
chunk_features = features_full[chunk_idx]   # (6, F, T_frames)

fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True, sharey=True)
titles = [
    r"$\tilde{I}_{a,x}$ — Active, front–back",
    r"$\tilde{I}_{a,y}$ — Active, left–right",
    r"$\tilde{I}_{a,z}$ — Active, up–down",
    r"$\tilde{I}_{r,x}$ — Reactive, front–back",
    r"$\tilde{I}_{r,y}$ — Reactive, left–right",
    r"$\tilde{I}_{r,z}$ — Reactive, up–down",
]

freq_hz = np.linspace(0, 8_000, preprocessor.n_freq)
T_frames = chunk_features.shape[-1]
time_s = np.linspace(0, preprocessor.chunk_duration, T_frames)

vmax = float(chunk_features.abs().quantile(0.99))

for ax, feat_ch, title in zip(axes.flat, chunk_features, titles):
    img = ax.imshow(
        feat_ch.numpy(),
        aspect="auto",
        origin="lower",
        extent=[time_s[0], time_s[-1], freq_hz[0] / 1000, freq_hz[-1] / 1000],
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax,
    )
    ax.set_title(title, fontsize=9)
    ax.set_ylabel("Frequency (kHz)")
    ax.set_xlabel("Time (s)")
    fig.colorbar(img, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle(
    f"Active & Reactive intensity — chunk {chunk_idx}  "
    f"az={np.degrees(azimuth.item()):.1f}°  "
    f"el={np.degrees(elevation.item()):.1f}°",
    fontsize=11,
)
plt.tight_layout()
plt.show()

In [ ]:
# ── Quick sanity check: time-averaged DOA estimate from intensity ─────────────
# Average active intensity over all TF bins → dominant DOA direction vector
Ia = features_full[:, :3, :, :]          # (N_chunks, 3, F, T_f) — active only
Ia_mean = Ia.mean(dim=(0, 2, 3))         # (3,)  mean over chunks, freq, time

az_est   = float(torch.atan2(Ia_mean[1], Ia_mean[0]))     # arctan2(Ia_y, Ia_x)
el_est   = float(torch.atan2(Ia_mean[2],
                              (Ia_mean[0]**2 + Ia_mean[1]**2).sqrt()))  # arctan2(Ia_z, |Ia_xy|)

az_true  = float(azimuth)
el_true  = float(elevation)

print("Intensity-based DOA estimate (no model, physics only)")
print(f"  Azimuth   — estimated: {np.degrees(az_est):+.1f}°   true: {np.degrees(az_true):+.1f}°  "
      f"  error: {abs(np.degrees(az_est - az_true)):.1f}°")
print(f"  Elevation — estimated: {np.degrees(el_est):+.1f}°   true: {np.degrees(el_true):+.1f}°  "
      f"  error: {abs(np.degrees(el_est - el_true)):.1f}°")
print()
print("Note: residual error is expected — this naive average ignores reverb,")
print("noise, and frequency weighting.  The neural network learns those.")

### 4.4 Integrating the Preprocessor with the Dataset

Pass the preprocessor as the `transform` argument to
`SpatialLibriSpeechDataset` so that feature extraction happens
transparently inside `__getitem__` on the `DataLoader` worker processes.

```python
from src.dataset    import SpatialLibriSpeechDataset
from src.preprocess import ActiveReactivePreprocessor

preprocessor = ActiveReactivePreprocessor(
    sample_rate=16_000,
    chunk_duration=0.5,
    n_fft=512,
    hop_length=128,
)

train_ds = SpatialLibriSpeechDataset(
    audio_root="/data/spatial_librispeech/ambisonics",
    split="train",
    transform=preprocessor,           # ← injected here
)

# Now __getitem__ returns (features, azimuth, elevation)
# where features.shape == (N_chunks, 6, 257, 63)
```

With `num_workers > 0` each worker runs `chunk → STFT → active/reactive`
in parallel on the CPU, fully overlapping I/O and compute with the GPU
forward pass.

---

## Summary

**Part 1** covered:
* Full documentation of the Spatial LibriSpeech parquet schema.
* Physical motivation for 3D Source Localization and the bicomplex inductive bias.
* Instantiation and inspection of `SpatialLibriSpeechDataset`, including
  angle distributions, waveform visualisation, and DataLoader compatibility.

**Part 2** covered:
* Physics of active and reactive acoustic intensity (Jacobsen & Rydén, 1990)
  and why these 6 features make DOA directly accessible to the network.
* STFT configuration choices and their spectro-temporal trade-offs.
* CPU-side 0.5 s chunking strategy to keep peak VRAM within the 16 GB budget.
* Demonstration of `ActiveReactivePreprocessor` on a live dataset sample,
  including intensity maps and a physics-only DOA sanity check.

**Next** → Notebook 02 will build the model zoo (`RealCNN` baseline and
`BiComplexNet`), define the training loop, and run the first localization
experiments.
